In [1]:
from pyspark.sql import SparkSession
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()

spark = SparkSession.builder.config(conf=sc.getConf()).getOrCreate()

glueContext = GlueContext(sc)

print("Spark inicializado com sucesso!")
print(spark.version)
print("GlueContext inicializado com sucesso!")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 97ece512-3060-4769-9675-11a5b5a31f67
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 97ece512-3060-4769-9675-11a5b5a31f67 to get into ready status...
Session 97ece512-3060-4769-9675-11a5b5a31f67 has been created.
Spark inicializado com sucesso!
3.3.0-amzn-1
GlueContext inicializado com sucesso!


In [2]:
import re
import unicodedata

from pyspark.sql import functions as F
from awsglue.dynamicframe import DynamicFrame

In [3]:
spark.sql("""
    CREATE DATABASE IF NOT EXISTS tech_challenge_silver
""")

print("Database criado ou já existente!")

DATABASE_SILVER = "tech_challenge_silver"

Database criado ou já existente!


In [4]:
# CONFIGURAÇÕES DA CAMADA SILVER

DATABASE_BRONZE = "tech-challenge-bronze"

SILVER_PREFIX = (
    "s3://lab-teste-tech-challenge-6890/Silver/"
)

PESQUISAS = [
    {
        "tabela": "tb_sod_2023_2024",
        "ano_pesquisa": "2023-2024",
        "destino": "sod_2023_2024"
    },
    {
        "tabela": "tb_sod_2024_2025",
        "ano_pesquisa": "2024-2025",
        "destino": "sod_2024_2025"
    },
    {
        "tabela": "tb_sod_2025_2026",
        "ano_pesquisa": "2025-2026",
        "destino": "sod_2025_2026"
    }
]

print("Etapa de configuração concluída com sucesso!")

Etapa de configuração concluída com sucesso!


In [5]:
# PADRONIZAÇÃO DOS NOMES DAS COLUNAS

def normalizar_nome_coluna(nome):
    """
    Remove acentos e substitui caracteres especiais
    """

    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ascii", "ignore").decode("ascii")
    nome = nome.lower()

    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = nome.strip("_")

    if not nome:
        nome = "coluna"

    if nome[0].isdigit():
        nome = f"col_{nome}"

    return nome


def padronizar_colunas(df):
    """
    Evita nomes duplicados
    """

    novos_nomes = []
    nomes_usados = set()

    for coluna in df.columns:
        nome_base = normalizar_nome_coluna(coluna)
        novo_nome = nome_base
        contador = 2

        while novo_nome in nomes_usados:
            novo_nome = f"{nome_base}_{contador}"
            contador += 1

        nomes_usados.add(novo_nome)
        novos_nomes.append(novo_nome)

    return df.toDF(*novos_nomes)

print("Etapa de padronização concluída com sucesso!")

Etapa de padronização concluída com sucesso!


In [6]:
# LIMPEZA DOS DADOS

def limpar_dados(df):
    # Remover espaços e tornar strings vazias em NULL
    for coluna, tipo in df.dtypes:
        if tipo == "string":
            valor = F.trim(F.col(coluna))

            df = df.withColumn(
                coluna,
                F.when(
                    valor == "",
                    F.lit(None).cast("string")
                ).otherwise(valor)
            )

    # Remover registros duplicados.
    df = df.dropDuplicates()

    return df

print("Etapa de limpeza concluída com sucesso!")


Etapa de limpeza concluída com sucesso!


In [7]:
# LEITURA E TRATAMENTO

def processar_silver(config):
    tabela_bronze = config["tabela"]
    ano_pesquisa = config["ano_pesquisa"]
    nome_destino = config["destino"]

    print(f"Iniciando processamento: {tabela_bronze}")

    # Ler a tabela Bronze
    nome_tabela = f"`{DATABASE_BRONZE}`.`{tabela_bronze}`"
    df = spark.table(nome_tabela)

    quantidade_inicial = df.count()
    print(f"Registros na Bronze: {quantidade_inicial}")

    # Padronizar nomes das colunas
    df = padronizar_colunas(df)

    # Limpar duplicidades
    df = limpar_dados(df)

    # Adicionar coluna ano_pesquisa
    df = df.withColumn(
        "ano_pesquisa",
        F.lit(ano_pesquisa)
    )

    print("Tratamento dos dados concluído!")

    return df, quantidade_inicial


In [29]:

# PROCESSAR SOD 2023-2024

df, quantidade_inicial = processar_silver(PESQUISAS[0])

# Definir o ano correto da pesquisa
df = df.withColumn(
    "ano_pesquisa",
    F.lit("2023-2024")
)

print(f"Registros na Bronze: {quantidade_inicial}")
print(f"Registros processados: {df.count()}")
print(f"Colunas geradas: {len(df.columns)}")

# Validar o ano da pesquisa
df.groupBy("ano_pesquisa").count().show()


Iniciando processamento: tb_sod_2023_2024
Registros na Bronze: 5293
Tratamento dos dados concluído!
Registros na Bronze: 5293
Registros processados: 5293
Colunas geradas: 400
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|   2023-2024| 5293|
+------------+-----+


In [30]:

# GRAVAR TABELA 2023-2024 NO S3 E GLUE CATALOG

destino_s3 = f"{SILVER_PREFIX}{PESQUISAS[0]['destino']}/"

sink = glueContext.getSink(
    path=destino_s3,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=[],
    compression="snappy",
    enableUpdateCatalog=True,
    transformation_ctx="s3output_sod_2023_2024"
)

sink.setCatalogInfo(
    catalogDatabase=DATABASE_SILVER,
    catalogTableName=f"tb_{PESQUISAS[0]['destino']}"
)

sink.setFormat("glueparquet")

sink.writeFrame(
    DynamicFrame.fromDF(
        df,
        glueContext,
        "df_silver_sod_2023_2024"
    )
)

print(f"Dados gravados em: {destino_s3}")
print(f"Tabela catalogada: {DATABASE_SILVER}.tb_{PESQUISAS[0]['destino']}")
print(f"Quantidade de linhas enviadas: {df.count()}")
print("Processamento concluído!")


Dados gravados em: s3://lab-teste-tech-challenge-6890/Silver/sod_2023_2024/
Tabela catalogada: tech_challenge_silver.tb_sod_2023_2024
Quantidade de linhas enviadas: 5293
Processamento concluído!


In [17]:

# PROCESSAR SOD 2024-2025

df, quantidade_inicial = processar_silver(PESQUISAS[1])

# Definir o ano correto da pesquisa
df = df.withColumn(
    "ano_pesquisa",
    F.lit("2024-2025")
)

print(f"Registros na Bronze: {quantidade_inicial}")
print(f"Registros processados: {df.count()}")
print(f"Colunas geradas: {len(df.columns)}")

# Validar o ano da pesquisa
df.groupBy("ano_pesquisa").count().show()


Iniciando processamento: tb_sod_2024_2025
Registros na Bronze: 5217
Tratamento dos dados concluído!
Registros na Bronze: 5217
Registros processados: 5215
Colunas geradas: 404
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|   2024-2025| 5215|
+------------+-----+


In [19]:

# GRAVAR TABELA 2024-2025 NO S3 E GLUE CATALOG

destino_s3 = f"{SILVER_PREFIX}{PESQUISAS[1]['destino']}/"

sink = glueContext.getSink(
    path=destino_s3,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=[],
    compression="snappy",
    enableUpdateCatalog=True,
    transformation_ctx="s3output_sod_2024_2025"
)

sink.setCatalogInfo(
    catalogDatabase=DATABASE_SILVER,
    catalogTableName=f"tb_{PESQUISAS[1]['destino']}"
)

sink.setFormat("glueparquet")

sink.writeFrame(
    DynamicFrame.fromDF(
        df,
        glueContext,
        "df_silver_sod_2024_2025"
    )
)

print(f"Dados gravados em: {destino_s3}")
print(f"Tabela catalogada: {DATABASE_SILVER}.tb_{PESQUISAS[1]['destino']}")
print(f"Quantidade de linhas enviadas: {df.count()}")
print("Processamento concluído!")


Dados gravados em: s3://lab-teste-tech-challenge-6890/Silver/sod_2024_2025/
Tabela catalogada: tech_challenge_silver.tb_sod_2024_2025
Quantidade de linhas enviadas: 5215
Processamento concluído!


In [20]:
# PROCESSAR SOD 2024-2025

df, quantidade_inicial = processar_silver(PESQUISAS[2])

# Definir o ano correto da pesquisa
df = df.withColumn(
    "ano_pesquisa",
    F.lit("2025-2026")
)

print(f"Registros na Bronze: {quantidade_inicial}")
print(f"Registros processados: {df.count()}")
print(f"Colunas geradas: {len(df.columns)}")

# Validar o ano da pesquisa
df.groupBy("ano_pesquisa").count().show()


Iniciando processamento: tb_sod_2025_2026
Registros na Bronze: 3495
Tratamento dos dados concluído!
Registros na Bronze: 3495
Registros processados: 3494
Colunas geradas: 389
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|   2025-2026| 3494|
+------------+-----+


In [23]:

# GRAVAR TABELA 2025-2026 NO S3 E GLUE CATALOG

destino_s3 = f"{SILVER_PREFIX}{PESQUISAS[2]['destino']}/"

sink = glueContext.getSink(
    path=destino_s3,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=[],
    compression="snappy",
    enableUpdateCatalog=True,
    transformation_ctx="s3output_sod_2025_2026"
)

sink.setCatalogInfo(
    catalogDatabase=DATABASE_SILVER,
    catalogTableName=f"tb_{PESQUISAS[2]['destino']}"
)

sink.setFormat("glueparquet")

sink.writeFrame(
    DynamicFrame.fromDF(
        df,
        glueContext,
        "df_silver_sod_2025_2026"
    )
)

print(f"Dados gravados em: {destino_s3}")
print(f"Tabela catalogada: {DATABASE_SILVER}.tb_{PESQUISAS[2]['destino']}")
print(f"Quantidade de linhas enviadas: {df.count()}")
print("Processamento concluído!")


Dados gravados em: s3://lab-teste-tech-challenge-6890/Silver/sod_2025_2026/
Tabela catalogada: tech_challenge_silver.tb_sod_2025_2026
Quantidade de linhas enviadas: 3494
Processamento concluído!


In [24]:

# PROCESSAR SOD 2025-2026

df, quantidade_inicial = processar_silver(PESQUISAS[2])

# Definir o ano correto da pesquisa
df = df.withColumn(
    "ano_pesquisa",
    F.lit("2025-2026")
)

print(f"Registros na Bronze: {quantidade_inicial}")
print(f"Registros processados: {df.count()}")
print(f"Colunas geradas: {len(df.columns)}")

# Validar o ano da pesquisa
df.groupBy("ano_pesquisa").count().show()


Iniciando processamento: tb_sod_2025_2026
Registros na Bronze: 3495
Tratamento dos dados concluído!
Registros na Bronze: 3495
Registros processados: 3494
Colunas geradas: 389
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|   2025-2026| 3494|
+------------+-----+


In [27]:

# GRAVAR TABELA 2025-2026 NO S3 E GLUE CATALOG

destino_s3 = f"{SILVER_PREFIX}{PESQUISAS[2]['destino']}/"

sink = glueContext.getSink(
    path=destino_s3,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=[],
    compression="snappy",
    enableUpdateCatalog=True,
    transformation_ctx="s3output_sod_2025_2026"
)

sink.setCatalogInfo(
    catalogDatabase=DATABASE_SILVER,
    catalogTableName=f"tb_{PESQUISAS[2]['destino']}"
)

sink.setFormat("glueparquet")

sink.writeFrame(
    DynamicFrame.fromDF(
        df,
        glueContext,
        "df_silver_sod_2025_2026"
    )
)

print(f"Dados gravados em: {destino_s3}")
print(f"Tabela catalogada: {DATABASE_SILVER}.tb_{PESQUISAS[2]['destino']}")
print(f"Quantidade de linhas enviadas: {df.count()}")
print("Processamento concluído!")


Dados gravados em: s3://lab-teste-tech-challenge-6890/Silver/sod_2025_2026/
Tabela catalogada: tech_challenge_silver.tb_sod_2025_2026
Quantidade de linhas enviadas: 3494
Processamento concluído!


In [26]:

# VALIDAR TABELA SILVER 2025-2026

tabela_silver = f"{DATABASE_SILVER}.tb_{PESQUISAS[2]['destino']}"

df_validacao = spark.table(tabela_silver)

print(f"Quantidade de linhas na tabela: {df_validacao.count()}")

df_validacao.groupBy("ano_pesquisa").count().show()


Quantidade de linhas na tabela: 3494
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|   2025-2026| 3494|
+------------+-----+
